# Stage 0d - gate check across candidate SAE scales (Kaggle GPU T4 x2, Internet ON). About 15-25 min. Paste the lines starting with ===== and 'gate score mean' (5 variants x 2 lines). Paper (2B) opens the gate on 98.7% of harmful and 4.7% of negative eval prompts.

In [ ]:
import os, subprocess, sys
REPO = "https://github.com/sarrazer24/reins-mini"
if not os.path.isdir("/kaggle/working/reins-mini"):
    !git clone {REPO} /kaggle/working/reins-mini
%cd /kaggle/working/reins-mini
!pip install -q -U transformers accelerate huggingface_hub
!git clone -q --depth 1 https://github.com/Geralt1020/REINS /tmp/REINS      # authors' data + frozen controllers; deleted at the end
GUISE = "/tmp/REINS/guise"; CTRL = "/tmp/REINS/configs/qwen35_2b_guise.controllers.json"
from huggingface_hub import snapshot_download
SAE_ROOT = snapshot_download("Carlos4869/REINS-SAE", allow_patterns=["qwen3.5-2b-base/*"], local_dir="/tmp/sae") + "/qwen3.5-2b-base"
print(os.listdir(SAE_ROOT)[:5])

In [ ]:
import subprocess, glob
variants = {"none": None, "nmse_min": "results/sae_scales_candidate_fine.json"}
for mult in (0.8, 1.0, 1.25):
    f = f"results/sae_scales_L0_128_x{mult}.json"
    subprocess.run(f"python scripts/07_make_scales.py --mult {mult} --out {f}", shell=True, check=True, capture_output=True)
    variants[f"L0_128_x{mult}"] = f
for name, f in variants.items():
    print("=====", name, flush=True)
    sc = f"--sae-scales {f}" if f else ""
    subprocess.run(f"python -u scripts/14_gate_scores.py --sae-root {SAE_ROOT} --controllers {CTRL} --guise-dir {GUISE} --n-per-category 0 {sc} --out runs/gatecheck/{name}.csv", shell=True, check=True)

In [ ]:
!rm -rf /tmp/REINS /tmp/sae